In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import wandb
import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B"
TRAINING_SET = "../../data/hannah_train_v1.jsonl"
CV_SET = "../../data/hannah_val_v1.jsonl"
OUTPUT_DIR = "./output/hannah-v3"

In [ ]:
import gc

# Clear peft model and trainer but NOT base_model — base weights stay in VRAM
if 'model' in dir():
    del model
if 'trainer' in dir():
    del trainer

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

In [ ]:
# ── Run once per session ──────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'system' %}<|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}"

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

In [ ]:
# ── Re-run this and everything below to retrain with new settings ──
# Strip any existing LoRA adapter so base_model stays clean
if 'model' in dir() and hasattr(model, 'peft_config'):
    _base = model.get_base_model()
else:
    _base = base_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(_base, lora_config)
model.print_trainable_parameters()

model.enable_input_require_grads()
model.config.use_cache = False

In [7]:
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAINING_SET,
        "cv": CV_SET
    }
)

print(dataset)

Generating train split: 420 examples [00:00, 13334.10 examples/s]
Generating cv split: 48 examples [00:00, 3429.05 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 420
    })
    cv: Dataset({
        features: ['messages'],
        num_rows: 48
    })
})


In [ ]:
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    bf16=True,
    fp16=False,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    save_total_limit=None,
    load_best_model_at_end=True,
    report_to="wandb",
    max_length=4096,
    completion_only_loss=True,
)

In [ ]:
import wandb

wandb.finish()

wandb.init(
    project="suicidal-llm",
    name="hannah-v3-4096ctx-r8",
    config={
        "model": MODEL_NAME,
        "rank": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "epochs": training_config.num_train_epochs,
        "batch_size": training_config.per_device_train_batch_size,
        "grad_accum": training_config.gradient_accumulation_steps,
        "effective_batch": training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps,
        "max_length": training_config.max_length,
        "learning_rate": training_config.learning_rate,
    }
)

In [11]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["cv"],
    processing_class=tokenizer,
)

Tokenizing eval dataset: 100%|██████████| 48/48 [00:00<00:00, 203.13 examples/s]


In [12]:
# Loss diagnostic — run this before trainer.train() to confirm loss is real
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
batch = {k: v.to("cuda") for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(f"Loss: {outputs.loss.item()}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

Loss: 3.106415033340454
VRAM used: 6.79 GB
VRAM free: 10.38 GB


In [13]:
print("Starting training...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Starting training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacity of 16.00 GiB of which 0 bytes is free. Of the allocated memory 22.20 GiB is allocated by PyTorch, and 89.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Save adapter weights
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
from huggingface_hub import login

HF_REPO = "remuspoon/hannah-v3"

login()

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"Pushed to https://huggingface.co/{HF_REPO}")

In [ ]:
from huggingface_hub import HfApi

HF_CHECKPOINTS_REPO = "remuspoon/hannah-v3-checkpoints"

api = HfApi()
api.create_repo(HF_CHECKPOINTS_REPO, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_CHECKPOINTS_REPO,
    repo_type="model",
)

print(f"All checkpoints pushed to https://huggingface.co/{HF_CHECKPOINTS_REPO}")